# Data Cleaning — Messy Employee Dataset
**Oasis Infobyte SIP — Data Analytics Track, Level 1, Task 3**

**Author:** Kuldeep Singh

**Objective:** Take a deliberately messy dataset and systematically transform it into a clean, analysis-ready dataset, documenting every decision.

**Note on the dataset:** A synthetic 'messy HR/employee records' dataset (`messy_employee_data.csv`) was generated for this notebook, deliberately seeded with the kinds of problems real-world exports have: nulls, duplicates, inconsistent category labels, mixed date formats, and outliers — so every cleaning technique below is demonstrated end-to-end.

In [ ]:

import numpy as np
import pandas as pd

np.random.seed(7)

n = 500
names = [f"Employee_{i}" for i in range(1, n+1)]
genders_clean = np.random.choice(["Male","Female"], n)
# messy gender labels
gender_variants = {"Male": ["Male","male","M","MALE"], "Female": ["Female","female","F","FEMALE"]}
gender = [np.random.choice(gender_variants[g]) for g in genders_clean]

dept = np.random.choice(["Sales","IT","HR","Finance","Operations"], n)

age = np.random.randint(21, 60, n).astype(float)
salary = np.random.normal(45000, 15000, n).round(2)
# inject some clearly wrong outlier salaries
salary[np.random.choice(n, 6, replace=False)] = np.random.choice([500000, -5000, 999999], 6)

join_dates = pd.date_range("2015-01-01", "2023-12-31", periods=n)
# mixed date string formats
date_strs = []
for d in join_dates:
    fmt = np.random.choice(["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y"])
    date_strs.append(d.strftime(fmt))

df = pd.DataFrame({
    "EmployeeName": names,
    "Gender": gender,
    "Department": dept,
    "Age": age,
    "Salary": salary,
    "JoinDate": date_strs
})

# inject nulls
for col, frac in [("Age", 0.04), ("Salary", 0.03), ("Department", 0.02), ("JoinDate", 0.02)]:
    idx = np.random.choice(n, int(n*frac), replace=False)
    df.loc[idx, col] = np.nan

# inject duplicate rows
dupes = df.sample(15, random_state=1)
df = pd.concat([df, dupes], ignore_index=True)

df.to_csv("messy_employee_data.csv", index=False)
print("Messy dataset generated:", df.shape)
df.head(8)


Messy dataset generated: (515, 6)


## 1. Data Quality Report (Before Cleaning)

In [ ]:

df = pd.read_csv("messy_employee_data.csv")

quality_before = pd.DataFrame({
    "nulls": df.isnull().sum(),
    "dtype": df.dtypes
})
print("Shape:", df.shape)
print("\nNulls & dtypes per column:\n", quality_before)
print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique Gender values (messy):", df["Gender"].unique())
print("Age range:", df["Age"].min(), "-", df["Age"].max())
print("Salary range:", df["Salary"].min(), "-", df["Salary"].max())


Shape: (515, 6)

Nulls & dtypes per column:
               nulls    dtype
EmployeeName      0      str
Gender            0      str
Department       10      str
Age              21  float64
Salary           15  float64
JoinDate         10      str

Duplicate rows: 15

Unique Gender values (messy): <StringArray>
['Female', 'male', 'FEMALE', 'MALE', 'F', 'M', 'Male', 'female']
Length: 8, dtype: str
Age range: 21.0 - 59.0
Salary range: -5000.0 - 500000.0


**Findings:** Nulls in `Age`, `Salary`, `Department`, `JoinDate`. 15 duplicate rows. `Gender` has 8 inconsistent label variants for 2 real categories. `Salary` has clearly invalid values (negative, and unrealistically large). `JoinDate` is stored as text in three different formats.

## 2. Duplicate Removal

In [ ]:

dupe_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {dupe_count} duplicate rows. New shape: {df.shape}")


Removed 15 duplicate rows. New shape: (500, 6)


**Decision:** Exact duplicate rows (identical across every column) are almost certainly the same record exported twice, so they were dropped outright rather than flagged for review.

## 3. Standardising Categorical Values

In [ ]:

print("Before:", sorted(df['Gender'].dropna().unique()))
df["Gender"] = df["Gender"].str.strip().str.upper().map({"MALE": "Male", "M": "Male",
                                                          "FEMALE": "Female", "F": "Female"})
print("After:", sorted(df['Gender'].dropna().unique()))


Before: ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male']
After: ['Female', 'Male']


**Decision:** All Gender variants (`Male`, `male`, `M`, `MALE`, etc.) were mapped to two canonical labels, `Male`/`Female`, so the column can be grouped/analysed correctly.

## 4. Fixing Date Formats

In [ ]:

def parse_mixed_date(val):
    if pd.isnull(val):
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y"):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(val, errors="coerce")  # fallback

df["JoinDate"] = df["JoinDate"].apply(parse_mixed_date)
print("JoinDate dtype now:", df["JoinDate"].dtype)
print("Unparseable dates left:", df["JoinDate"].isnull().sum() - df["JoinDate"].isnull().sum())
df[["JoinDate"]].head()


JoinDate dtype now: datetime64[us]
Unparseable dates left: 0


**Decision:** Each of the three date formats found in the raw text (`YYYY-MM-DD`, `DD/MM/YYYY`, `DD-Mon-YYYY`) is tried explicitly rather than relying on pandas' format guessing, which avoids silently mis-parsing day/month for ambiguous dates.

## 5. Outlier Detection — Salary (IQR Method)

In [ ]:

Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df["Salary"] < lower) | (df["Salary"] > upper)]
print(f"IQR bounds: [{lower:.0f}, {upper:.0f}]")
print(f"Outliers found: {len(outliers)}")
print(outliers[["EmployeeName","Salary"]])


IQR bounds: [1781, 87792]
Outliers found: 8
     EmployeeName     Salary
65    Employee_66   -3832.73
69    Employee_70  500000.00
97    Employee_98   -5000.00
133  Employee_134   -1150.81
228  Employee_229   -5000.00
392  Employee_393   90110.02
451  Employee_452  500000.00
487  Employee_488   -5000.00


In [ ]:

# Decision: negative/absurd salaries (data entry errors) are treated as missing and imputed;
# genuinely high-but-plausible salaries would be capped, but here the extreme values
# (500000, -5000, 999999) are clear entry errors, so we null them and impute with the median.
df.loc[(df["Salary"] < 0) | (df["Salary"] > 200000), "Salary"] = np.nan
print("Outliers nulled. Remaining nulls in Salary:", df["Salary"].isnull().sum())


Outliers nulled. Remaining nulls in Salary: 22


**Decision documented:** Salaries below 0 or above ₹200,000/month are not plausible for this workforce, so they were treated as data-entry errors (nulled) rather than real high earners, then imputed like any other missing value in the next step.

## 6. Missing Data Handling

In [ ]:

# Age: numeric, roughly symmetric -> median imputation
df["Age"] = df["Age"].fillna(df["Age"].median())

# Salary: numeric, imputed with median (robust to remaining skew)
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

# Department: categorical -> mode imputation
df["Department"] = df["Department"].fillna(df["Department"].mode()[0])

# JoinDate: no reliable way to guess a missing join date -> drop those rows
before_rows = len(df)
df = df.dropna(subset=["JoinDate"]).reset_index(drop=True)
print(f"Dropped {before_rows - len(df)} rows with missing JoinDate (no safe way to impute a date).")
print("\nRemaining nulls:\n", df.isnull().sum())


Dropped 10 rows with missing JoinDate (no safe way to impute a date).

Remaining nulls:
 EmployeeName    0
Gender          0
Department      0
Age             0
Salary          0
JoinDate        0
dtype: int64


**Decisions documented:**
- `Age`, `Salary` → median imputation (robust to skew/outliers, keeps all rows).
- `Department` → mode imputation (categorical, no ordering to interpolate).
- `JoinDate` → rows dropped, since a wrong guessed join date would corrupt tenure calculations and there's no reliable signal to infer it from.

## 7. Data Type Correction

In [ ]:

df["Age"] = df["Age"].astype(int)
df["Salary"] = df["Salary"].round(2)
df["EmployeeName"] = df["EmployeeName"].astype(str)
print(df.dtypes)


EmployeeName               str
Gender                     str
Department                 str
Age                      int64
Salary                 float64
JoinDate        datetime64[us]
dtype: object


## 8. Before vs. After Summary

In [ ]:

raw = pd.read_csv("messy_employee_data.csv")

summary = pd.DataFrame({
    "Metric": ["Row count", "Duplicate rows", "Total nulls", "Gender categories", "Salary min", "Salary max"],
    "Before": [
        len(raw), raw.duplicated().sum(), raw.isnull().sum().sum(),
        raw["Gender"].nunique(), raw["Salary"].min(), raw["Salary"].max()
    ],
    "After": [
        len(df), df.duplicated().sum(), df.isnull().sum().sum(),
        df["Gender"].nunique(), df["Salary"].min(), df["Salary"].max()
    ]
})
print(summary.to_string(index=False))

df.to_csv("cleaned_employee_data.csv", index=False)
print("\nCleaned dataset saved as cleaned_employee_data.csv")


           Metric   Before    After
        Row count    515.0   490.00
   Duplicate rows     15.0     0.00
      Total nulls     56.0     0.00
Gender categories      8.0     2.00
       Salary min  -5000.0  4745.67
       Salary max 500000.0 90110.02

Cleaned dataset saved as cleaned_employee_data.csv


## Conclusion

The raw export had duplicate rows, inconsistent category labels, mixed date formats, invalid outlier salaries, and missing values across four columns. Each issue was resolved with an explicit, documented strategy (drop duplicates, standardise categories, explicit multi-format date parsing, IQR-based outlier nulling, median/mode imputation) rather than a blanket approach, and the before/after summary confirms the dataset is now duplicate-free, null-free, and has consistent categories and valid numeric ranges — ready for downstream analysis.